# LangSmith 기초 - LLM 앱 추적과 관찰

## LangSmith란

LangSmith는 LangCahin/LangGraph 앱의 모든 실행(LLM 호출, 프롬프트, 도구 사용)을 자동으로 기록하고 웹 대시보드에서 확인하게 해주는 **관찰 플랫폼**이다.

## .env 파일 예시

```
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 설정되지 않았다."

# .env에 과거 표기(LANGCHAIN_API_KEY)로 저장했어도 동작하도록 처리
langsmith_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
assert langsmith_key, "LANGSMITH_API_KEY가 .env에 설정되지 않았다."

os.environ["LANGSMITH_API_KEY"] = langsmith_key
os.environ["LANGSMITH_TRACING"] = "true"          # 이 한 줄로 모든 실행이 추적된다
os.environ["LANGSMITH_PROJECT"] = "langsmith-basic"  # 대시보드에서 실행을 묶는 단위

print("LangSmith 추적 활성화 -> 프로젝트: langsmith-basic")

LangSmith 추적 활성화 -> 프로젝트: langsmith-basic


## 1단계: 추적할 체인 준비

평범한 LCEL 체인이다. 추적을 위한 코드는 한 줄도 없다.

환경변수만으로 자동 추적된다.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 초보자에게도 쉽게 설명하는 AI 도우미입니다. 핵심을 3~5문장으로 설명하세요."),
    ("human", "질문: {question}"),
])

# chain 만들고 구동
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "파이썬이란?"})
print(result)

파이썬은 배우기 쉽고 사용하기 간편한 프로그래밍 언어입니다. 다양한 용도로 사용되며, 웹 개발, 데이터 분석, 인공지능 등 여러 분야에서 인기가 높습니다. 문법이 간단하고 명확하여 초보자도 쉽게 접근할 수 있습니다. 또한, 많은 라이브러리와 프레임워크가 있어 다양한 기능을 쉽게 구현할 수 있습니다.


위 셀을 실행한 뒤 https://smith.langchain.com 의 `langsmith-basic` 프로젝트를 열어본다.
Trace 하나를 클릭하면 `RunnableSequence → ChatPromptTemplate → ChatOpenAI → StrOutputParser`의 트리가 보이고, 각 단계의 입출력, 토큰 수, 소요 시간이 기록되어 있다.

## 2단계: run_name, tags, metadata로 실행 구분하기

실행이 쌓이면 검색과 필터링이 필요하다. `config`에 이름표를 붙인다.

| 항목 | 용도 |
|------|------|
| `run_name` | 대시보드에 표시될 실행 이름 |
| `tags` | 분류용 라벨 (필터: Tags) |
| `metadata` | key-value 상세 정보 (필터: Metadata) — 프롬프트 버전, 모델명 등 |

In [5]:
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(
    run_name = "qa_chain",
    tags = ["demo", "qa", "v1"],
    metadata = {"prompt_version": "v1", "model_name": "gpt-4o-mini"}
)

questions = [
    "파이썬이란?",
    "자바와 파이썬의 차이는?",
    "머신러닝을 한 문단으로 설명해줘.",
]

for q in questions:
    result = chain.invoke({"question": q}, config=config)
    print(f"[질문] {q}\n[답변] {result}\n")

[질문] 파이썬이란?
[답변] 파이썬은 배우기 쉽고 사용하기 편리한 프로그래밍 언어입니다. 다양한 분야에서 활용되며, 웹 개발, 데이터 분석, 인공지능 등 여러 용도로 사용됩니다. 간결한 문법 덕분에 초보자도 쉽게 접근할 수 있습니다. 또한, 많은 라이브러리와 커뮤니티 지원이 있어 개발에 유용합니다.

[질문] 자바와 파이썬의 차이는?
[답변] 자바와 파이썬은 두 가지 인기 있는 프로그래밍 언어입니다. 자바는 정적 타입 언어로, 변수를 사용하기 전에 타입을 명시해야 하며, 주로 대규모 애플리케이션에 적합합니다. 반면, 파이썬은 동적 타입 언어로, 코드가 간결하고 읽기 쉬워 초보자에게 인기가 많습니다. 또한, 파이썬은 다양한 라이브러리와 프레임워크를 제공하여 데이터 과학, 웹 개발 등 여러 분야에서 활용됩니다.

[질문] 머신러닝을 한 문단으로 설명해줘.
[답변] 머신러닝은 컴퓨터가 데이터를 통해 학습하고, 경험을 바탕으로 스스로 개선하는 기술입니다. 이를 통해 프로그램은 명시적인 지시 없이도 패턴을 인식하고 예측을 할 수 있습니다. 예를 들어, 이메일 필터링, 이미지 인식, 추천 시스템 등이 머신러닝의 활용 사례입니다. 머신러닝은 주로 데이터와 알고리즘을 사용하여 문제를 해결하는 데 중점을 둡니다.



## 3단계: 스트리밍도 동일하게 추적된다

In [6]:
stream_config = RunnableConfig(
    run_name = "qa_chain_stream",
    tags = ["demo", "qa", "stream"],
    metadata = {"prompt_version": "v1", "model_name": "gpt-4o-mini"}
)

for chunk in chain.stream({"question": "딥러닝이란?"}, config=stream_config):
    print(chunk, end="", flush=True)

딥러닝은 인공지능의 한 분야로, 컴퓨터가 데이터를 통해 스스로 학습하고 예측할 수 있도록 하는 기술입니다. 주로 인공신경망이라는 구조를 사용하여, 이미지 인식, 음성 인식, 자연어 처리 등 다양한 작업을 수행합니다. 딥러닝은 대량의 데이터와 강력한 컴퓨팅 파워를 활용하여 성능을 향상시키는 것이 특징입니다.

## 대시보드에서 확인할 것 — 관찰 포인트

`langsmith-basic` 프로젝트에서 다음 다섯 가지를 직접 찾아본다.

1. **Trace 트리 구조**: 체인의 각 단계(프롬프트 → LLM → 파서)가 어떤 순서로 실행됐고, 단계별 입출력이 무엇인지. 프롬프트 템플릿에 변수가 실제로 어떻게 채워졌는지 확인한다.
2. **토큰과 비용**: 실행마다 input/output 토큰 수와 예상 비용이 기록된다. 프로젝트 화면 상단에서 누적 비용도 보인다.
3. **지연 시간(Latency)**: 전체 실행 시간 중 어느 단계가 오래 걸렸는지. LLM 호출이 대부분을 차지하는 것이 정상이다.
4. **필터링**: 좌측 필터에서 Tags=`stream`으로 걸러 스트리밍 실행만 찾아본다. Metadata의 `prompt_version`으로도 필터가 가능하다.
5. **오류 추적**: 실패한 실행은 빨간색으로 표시되며, 어느 단계에서 어떤 예외가 났는지 그대로 보인다.

### 실습 과제

시스템 프롬프트를 수정하고(예: "10문장으로 상세히") `metadata={"prompt_version": "v2"}`로 다시 실행한 뒤, 대시보드에서 v1과 v2의 답변 품질·토큰 수·비용을 비교해본다. 이것이 **프롬프트 버전 관리와 A/B 비교**의 기본 패턴이다.
